# Raccolta domande di Matematica — PoliMillionaire

Gioca partite sulla competizione Maths, raccoglie le domande con la risposta corretta
e le salva in `math_questions.json` per usarle come dataset di fine-tuning.

In [1]:
import os
import sys
import random
import json


package_parent_dir = os.path.abspath('./')
if package_parent_dir not in sys.path:
    sys.path.append(package_parent_dir)

from millionaire_client import MillionaireClient, AuthenticationError
import dotenv
dotenv.load_dotenv()

username = os.getenv('USER')
pwd      = os.getenv('PASSWORD')
API_URL  = 'http://131.175.15.22:51111/'

client = MillionaireClient(API_URL)
try:
    user = client.login(username, pwd)
    print(f'Login successful. Welcome, {user.username}!')
except AuthenticationError as e:
    print(f'Login failed: {e}')

Login successful. Welcome, Alisss!


In [ ]:
import os, sys, json, random, re, ollama

package_parent_dir = os.path.abspath('./')
if package_parent_dir not in sys.path:
    sys.path.append(package_parent_dir)

from millionaire_client import MillionaireClient
import dotenv
dotenv.load_dotenv()

API_URL  = 'http://131.175.15.22:51111/'
client   = MillionaireClient(API_URL)
client.login(os.getenv('USER'), os.getenv('PASSWORD'))

MATHS_ID = 3
LABELS   = ['A', 'B', 'C', 'D']
OUT_FILE = 'math_questions.json'

# Carica domande già raccolte (se esistono)
if os.path.exists(OUT_FILE):
    with open(OUT_FILE) as f:
        collected = json.load(f)
    print(f'Caricate {len(collected)} domande esistenti.')
else:
    collected = []
    print('Nessuna domanda esistente, si parte da zero.')

seen_texts = {q['question'] for q in collected}


def _llm_answer(question):
    opts = '\n'.join(f'{LABELS[i]}) {opt.text}' for i, opt in enumerate(question.options))
    prompt = (
        'Answer the multiple choice question below.\n'
        'Reason briefly, then on the last line write exactly: Answer: X\n'
        '(X must be A, B, C, or D)\n\n'
        f'Question: {question.text}\n\n{opts}\n'
    )
    raw = ollama.chat(
        model='llama3.1:8b',
        messages=[{'role': 'user', 'content': prompt}],
        options={'num_predict': 300, 'temperature': 0},
    )['message']['content']
    upper = raw.strip().upper()
    m = re.search(r'ANSWER\s*[:=]\s*([A-D])', upper)
    if m:
        label = m.group(1)
    else:
        m = re.search(r'\b([A-D])\b\s*$', upper)
        label = m.group(1) if m else None
    if label is None:
        label = random.choice(LABELS)
    return question.options[LABELS.index(label)].id, label


def collect_game(game_index):
    game = client.game.start(MATHS_ID)
    new_questions = []

    while game.in_progress:
        q = game.current_question
        if not q:
            break

        chosen_id, chosen_label = _llm_answer(q)
        result = game.answer(chosen_id)

        if result.correct and q.text not in seen_texts:
            entry = {
                'question':      q.text,
                'options':       {LABELS[i]: o.text for i, o in enumerate(q.options)},
                'correct_label': chosen_label,
                'correct_text':  next(o.text for o in q.options if o.id == chosen_id),
                'level':         q.level,
            }
            new_questions.append(entry)
            seen_texts.add(q.text)

        if result.game_over:
            break

    print(f'  Partita {game_index}: +{len(new_questions)} domande nuove (livello: {game.current_level})')
    return new_questions


N_GAMES = 20

print(f'Avvio raccolta su {N_GAMES} partite...')
for i in range(1, N_GAMES + 1):
    new = collect_game(i)
    collected.extend(new)
    with open(OUT_FILE, 'w') as f:
        json.dump(collected, f, ensure_ascii=False, indent=2)

print(f'\nTotale domande raccolte: {len(collected)}')
print(f'Salvate in {OUT_FILE}')

In [ ]:
# Mostra le domande raccolte
with open(OUT_FILE) as f:
    qs = json.load(f)

print(f'Totale: {len(qs)} domande\n')
for i, q in enumerate(qs[:10]):  # mostra le prime 10
    print(f'[{i+1}] (livello {q["level"]}) {q["question"]}')
    for label, text in q['options'].items():
        marker = ' ✓' if label == q['correct_label'] else ''
        print(f'     {label}) {text}{marker}')
    print()

In [ ]:
# Statistiche per livello
from collections import Counter

levels = Counter(q['level'] for q in qs)
print('Distribuzione per livello:')
for lvl in sorted(levels):
    print(f'  Livello {lvl:2d}: {levels[lvl]:3d} domande')
print(f'\nTotale: {len(qs)}')

In [ ]:
# Esporta in formato pronto per il fine-tuning (chat template)
FINETUNE_FILE = 'math_finetune_dataset.json'

finetune_data = []
for q in qs:
    opts_str = '\n'.join(f"{label}) {text}" for label, text in q['options'].items())
    finetune_data.append({
        'messages': [
            {
                'role': 'user',
                'content': (
                    'Answer this multiple-choice math question.\n'
                    'Think step by step, then end with: Answer: X (X = A, B, C or D)\n\n'
                    f'Question: {q["question"]}\n\n{opts_str}'
                )
            },
            {
                'role': 'assistant',
                'content': f'Answer: {q["correct_label"]}'
            }
        ]
    })

with open(FINETUNE_FILE, 'w') as f:
    json.dump(finetune_data, f, ensure_ascii=False, indent=2)

print(f'Dataset fine-tuning salvato: {FINETUNE_FILE} ({len(finetune_data)} esempi)')